# Prefix-Tuned BERT vs. Text CNN: Final Model Comparison

**Goal:** prefix-tune a pretrained BERT model for text classification and compare it against the Text CNN and best RNN model from the earlier notebooks.
**Dataset:** the same text classification dataset as the other Project 3 notebooks (198-example validation split).
**Approach:** prefix-tuning (5-token prefix) of `bert-base-uncased`, then a final best-checkpoint comparison of Text CNN vs. LSTM-max on the held-out test set.
**Headline result:** prefix-tuned BERT reached only 72.22% validation accuracy — well below the Text CNN, which was selected as the final best model (97.01% test accuracy). See `README.md` and `../../docs/RESULTS.md`.

*Built as part of Monash University FIT5215 Deep Learning (2026).*


Author: Rahul Bhatt

# Prefix-Tuning BERT vs Text CNN and LSTM-max for Text Classification

This notebook is Part 4 of a deep learning assignment (FIT5215, Monash University) exploring sequential models for text classification. It implements a Transformer encoder from scratch, prefix-tunes a pretrained BERT model, and compares the result against a Text CNN and an LSTM-max model built in earlier notebooks.

## Setup

### <font color="#0b486b">Set random seeds</font>

We need to install the package datasets for creating BERT datasets.

In [1]:
!pip install datasets

We start with importing PyTorch and NumPy and setting random seeds for PyTorch and NumPy. You can use any seeds you prefer.

In [112]:
import os
import torch
import random
import requests
import pandas as pd
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import BertTokenizer
import os
from six.moves.urllib.request import urlretrieve
from sklearn import preprocessing
import matplotlib.pyplot as plt
plt.style.use('ggplot')

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [113]:
def seed_all(seed=1029):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_all(seed=1234)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## <font color="#0b486b">Download and preprocess the data</font>


The dataset we use for this assignment is a question classification dataset for which the training set consists of $5,500$ questions belonging to 6 coarse question categories including:
- abbreviation (ABBR),
- entity (ENTY),
- description (DESC),
- human (HUM),
- location (LOC) and
- numeric (NUM).

In this assignment, we will utilize a subset of this dataset, containing $2,000$ questions for training and validation. We will use 80% of those 2000 questions for trainning and the rest for validation.


The `TextDataManager` class handles downloading and preprocessing the question-classification dataset used throughout this notebook.

In [117]:
class DataManager:
    """
    This class manages and preprocesses a simple text dataset for a sentence classification task.

    Attributes:
        verbose (bool): Controls verbosity for printing information during data processing.
        max_sentence_len (int): The maximum length of a sentence in the dataset.
        str_questions (list): A list to store the string representations of the questions in the dataset.
        str_labels (list): A list to store the string representations of the labels in the dataset.
        numeral_labels (list): A list to store the numerical representations of the labels in the dataset.
        maxlen (int): Maximum length for padding sequences. Sequences longer than this length will be truncated,
            and sequences shorter than this length will be padded with zeros. Defaults to 50.
        numeral_data (list): A list to store the numerical representations of the questions in the dataset.
        random_state (int): Seed value for random number generation to ensure reproducibility.
            Set this value to a specific integer to reproduce the same random sequence every time. Defaults to 6789.
        random (np.random.RandomState): Random number generator object initialized with the given random_state.
            It is used for various random operations in the class.

    Methods:
        maybe_download(dir_name, file_name, url, verbose=True):
            Downloads a file from a given URL if it does not exist in the specified directory.
            The directory and file are created if they do not exist.

        read_data(dir_name, file_names):
            Reads data from files in a directory, preprocesses it, and computes the maximum sentence length.
            Each file is expected to contain rows in the format "<label>:<question>".
            The labels and questions are stored as string representations.

        manipulate_data():
            Performs data manipulation by tokenizing, numericalizing, and padding the text data.
            The questions are tokenized and converted into numerical sequences using a tokenizer.
            The sequences are padded or truncated to the maximum sequence length.

        train_valid_test_split(train_ratio=0.9):
            Splits the data into training, validation, and test sets based on a given ratio.
            The data is randomly shuffled, and the specified ratio is used to determine the size of the training set.
            The string questions, numerical data, and numerical labels are split accordingly.
            TensorFlow `Dataset` objects are created for the training and validation sets.


    """

    def __init__(self, verbose=True, random_state=6789):
        self.verbose = verbose
        self.max_sentence_len = 0
        self.str_questions = list()
        self.str_labels = list()
        self.numeral_labels = list()
        self.numeral_data = list()
        self.random_state = random_state
        self.random = np.random.RandomState(random_state)

    @staticmethod
    def maybe_download(dir_name, file_name, url, verbose=True):
        if not os.path.exists(dir_name):
            os.mkdir(dir_name)
        if not os.path.exists(os.path.join(dir_name, file_name)):
            urlretrieve(url + file_name, os.path.join(dir_name, file_name))
        if verbose:
            print("Downloaded successfully {}".format(file_name))

    def read_data(self, dir_name, file_names):
        self.str_questions = list()
        self.str_labels = list()
        for file_name in file_names:
            file_path= os.path.join(dir_name, file_name)
            with open(file_path, "r", encoding="latin-1") as f:
                for row in f:
                    row_str = row.split(":")
                    label, question = row_str[0], row_str[1]
                    question = question.lower()
                    self.str_labels.append(label)
                    self.str_questions.append(question[0:-1])
                    if self.max_sentence_len < len(self.str_questions[-1]):
                        self.max_sentence_len = len(self.str_questions[-1])

        # turns labels into numbers
        le = preprocessing.LabelEncoder()
        le.fit(self.str_labels)
        self.numeral_labels = np.array(le.transform(self.str_labels))
        self.str_classes = le.classes_
        self.num_classes = len(self.str_classes)
        if self.verbose:
            print("\nSample questions and corresponding labels... \n")
            print(self.str_questions[0:5])
            print(self.str_labels[0:5])

    def manipulate_data(self):
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        vocab = self.tokenizer.get_vocab()
        self.word2idx = {w: i for i, w in enumerate(vocab)}
        self.idx2word = {i:w for w,i in self.word2idx.items()}
        self.vocab_size = len(self.word2idx)

        token_ids = []
        num_seqs = []
        for text in self.str_questions:  # iterate over the list of text
          text_seqs = self.tokenizer.tokenize(str(text))  # tokenize each text individually
          # Convert tokens to IDs
          token_ids = self.tokenizer.convert_tokens_to_ids(text_seqs)
          # Convert token IDs to a tensor of indices using your word2idx mapping
          seq_tensor = torch.LongTensor(token_ids)
          num_seqs.append(seq_tensor)  # append the tensor for each sequence

        # Pad the sequences and create a tensor
        if num_seqs:
          self.numeral_data = pad_sequence(num_seqs, batch_first=True)  # Pads to max length of the sequences
          self.num_sentences, self.max_seq_len = self.numeral_data.shape

    def train_valid_test_split(self, train_ratio=0.8, test_ratio = 0.1):
        train_size = int(self.num_sentences*train_ratio) +1
        test_size = int(self.num_sentences*test_ratio) +1
        valid_size = self.num_sentences - (train_size + test_size)
        data_indices = list(range(self.num_sentences))
        random.shuffle(data_indices)
        self.train_str_questions = [self.str_questions[i] for i in data_indices[:train_size]]
        self.train_numeral_labels = self.numeral_labels[data_indices[:train_size]]
        train_set_data = self.numeral_data[data_indices[:train_size]]
        train_set_labels = self.numeral_labels[data_indices[:train_size]]
        train_set_labels = torch.from_numpy(train_set_labels)
        train_set = torch.utils.data.TensorDataset(train_set_data, train_set_labels)
        self.test_str_questions = [self.str_questions[i] for i in data_indices[-test_size:]]
        self.test_numeral_labels = self.numeral_labels[data_indices[-test_size:]]
        test_set_data = self.numeral_data[data_indices[-test_size:]]
        test_set_labels = self.numeral_labels[data_indices[-test_size:]]
        test_set_labels = torch.from_numpy(test_set_labels)
        test_set = torch.utils.data.TensorDataset(test_set_data, test_set_labels)
        self.valid_str_questions = [self.str_questions[i] for i in data_indices[train_size:-test_size]]
        self.valid_numeral_labels = self.numeral_labels[data_indices[train_size:-test_size]]
        valid_set_data = self.numeral_data[data_indices[train_size:-test_size]]
        valid_set_labels = self.numeral_labels[data_indices[train_size:-test_size]]
        valid_set_labels = torch.from_numpy(valid_set_labels)
        valid_set = torch.utils.data.TensorDataset(valid_set_data, valid_set_labels)
        self.train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
        self.test_loader = DataLoader(test_set, batch_size=64, shuffle=False)
        self.valid_loader = DataLoader(valid_set, batch_size=64, shuffle=False)

In [118]:
print('Loading data...')
DataManager.maybe_download("data", "train_2000.label", "http://cogcomp.org/Data/QA/QC/")

dm = DataManager()
dm.read_data("data/", ["train_2000.label"])

Loading data...
Downloaded successfully train_2000.label

Sample questions and corresponding labels... 

['manner how did serfdom develop in and then leave russia ?', 'cremat what films featured the character popeye doyle ?', "manner how can i find a list of celebrities ' real names ?", 'animal what fowl grabs the spotlight after the chinese year of the monkey ?', 'exp what is the full form of .com ?']
['DESC', 'ENTY', 'DESC', 'ENTY', 'ABBR']


In [119]:
dm.manipulate_data()
dm.train_valid_test_split(train_ratio=0.8, test_ratio = 0.1)

In [120]:
for x, y in dm.train_loader:
    print(x.shape, y.shape)
    break

torch.Size([64, 36]) torch.Size([64])


We now declare the `BaseTrainer` class, which will be used later to train the subsequent deep learning models for text data.

In [122]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class BaseTrainer:
    def __init__(self, model, criterion, optimizer, train_loader, val_loader):
        self.model = model
        self.criterion = criterion  #the loss function
        self.optimizer = optimizer  #the optimizer
        self.train_loader = train_loader  #the train loader
        self.val_loader = val_loader  #the valid loader

    #the function to train the model in many epochs
    def fit(self, num_epochs):
        self.num_batches = len(self.train_loader)

        for epoch in range(num_epochs):
            print(f'Epoch {epoch + 1}/{num_epochs}')
            train_loss, train_accuracy = self.train_one_epoch()
            val_loss, val_accuracy = self.validate_one_epoch()
            print(
                f'{self.num_batches}/{self.num_batches} - train_loss: {train_loss:.4f} - train_accuracy: {train_accuracy*100:.4f}% \
                - val_loss: {val_loss:.4f} - val_accuracy: {val_accuracy*100:.4f}%')

    #train in one epoch, return the train_acc, train_loss
    def train_one_epoch(self):
        self.model.train()
        running_loss, correct, total = 0.0, 0, 0
        for i, data in enumerate(self.train_loader):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)
            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        train_accuracy = correct / total
        train_loss = running_loss / self.num_batches
        return train_loss, train_accuracy

    #evaluate on a loader and return the loss and accuracy
    def evaluate(self, loader):
        self.model.eval()
        loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for data in loader:
                inputs, labels = data
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = correct / total
        loss = loss / len(self.val_loader)
        return loss, accuracy

    #return the val_acc, val_loss, be called at the end of each epoch
    def validate_one_epoch(self):
      val_loss, val_accuracy = self.evaluate(self.val_loader)
      return val_loss, val_accuracy

## Transformers: Encoder, Prefix-Tuned BERT, and Final Model Comparison

This section implements a Transformer encoder from scratch, prefix-tunes a pretrained BERT model for text classification, and compares the best models from across the assignment.

### Transformer Encoder for Text Classification

Implements the multi-head self-attention, position-wise feed-forward, and positional encoding modules of a Transformer encoder (Vaswani et al., 2017), using the encoder's output for text classification.

Below is the code of `MultiHeadSelfAttention`, `PositionWiseFeedForward`, `PositionalEncoding`, and `EncoderLayer`.

In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        # Ensure that the model dimension (d_model) is divisible by the number of heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        # Initialize dimensions
        self.d_model = d_model # Model's dimension
        self.num_heads = num_heads # Number of attention heads
        self.d_k = d_model // num_heads # Dimension of each head's key, query, and value

        # Linear layers for transforming inputs
        self.W_q = nn.Linear(d_model, d_model) # Query transformation
        self.W_k = nn.Linear(d_model, d_model) # Key transformation
        self.W_v = nn.Linear(d_model, d_model) # Value transformation
        self.W_o = nn.Linear(d_model, d_model) # Output transformation

    def scaled_dot_product_attention(self, Q, K, V):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Apply mask if provided (useful for preventing attention to certain parts like padding)
        #if mask is not None:
            #attn_scores = attn_scores.masked_fill(mask == 0, -1e9)

        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)

        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output

    def split_heads(self, x):
        # Reshape the input to have num_heads for multi-head attention
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q, K, V):
        # Apply linear transformations and split heads
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V)

        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output

In [10]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

In [11]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [12]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_output = self.self_attn(x, x, x)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

The `TransformerClassifier` passes token embeddings `[batch_size, seq_len, embed_dim]` through a stack of `EncoderLayer`s, averages the resulting embeddings across `seq_len`, and feeds the result into a linear classification layer.

In [13]:
class TransformerClassifier(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, num_layers, dropout_rate=0.2, data_manager = None):
        super(TransformerClassifier, self).__init__()
        self.vocab_size = data_manager.vocab_size
        self.num_classes = data_manager.num_classes
        self.embed_dim = embed_dim
        self.max_seq_len = data_manager.max_seq_len
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate

    def build(self):
        self.embedding = nn.Embedding(self.vocab_size, self.embed_dim)
        self.pos_encoding = PositionalEncoding(self.embed_dim, self.max_seq_len)
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(self.embed_dim, self.num_heads, self.ff_dim, self.dropout_rate)
            for _ in range(self.num_layers)
        ])
        self.dropout = nn.Dropout(self.dropout_rate)
        self.fc = nn.Linear(self.embed_dim, self.num_classes)

    def forward(self, x):
        e = self.embedding(x) * math.sqrt(self.embed_dim)
        e = self.pos_encoding(e)
        e = self.dropout(e)
        for layer in self.encoder_layers:
            e = layer(e)
        pooled = e.mean(dim=1)
        return self.fc(pooled)




In [14]:
transformer = TransformerClassifier(embed_dim=512, num_heads=8, ff_dim=2048, num_layers=12, dropout_rate=0.1, data_manager= dm)
transformer.build()
transformer = transformer.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(transformer.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)
trainer = BaseTrainer(model= transformer, criterion=criterion, optimizer=optimizer, train_loader=dm.train_loader, val_loader=dm.valid_loader)
trainer.fit(num_epochs=30)

Epoch 1/30
26/26 - train_loss: 1.9615 - train_accuracy: 22.6733%                 - val_loss: 0.9026 - val_accuracy: 12.6263%
Epoch 2/30
26/26 - train_loss: 1.6670 - train_accuracy: 25.8588%                 - val_loss: 0.7338 - val_accuracy: 37.3737%
Epoch 3/30
26/26 - train_loss: 1.2980 - train_accuracy: 42.5359%                 - val_loss: 0.6143 - val_accuracy: 47.4747%
Epoch 4/30
26/26 - train_loss: 1.3248 - train_accuracy: 42.2861%                 - val_loss: 0.6105 - val_accuracy: 48.4848%
Epoch 5/30
26/26 - train_loss: 0.9697 - train_accuracy: 61.0868%                 - val_loss: 0.1494 - val_accuracy: 83.3333%
Epoch 6/30
26/26 - train_loss: 0.4895 - train_accuracy: 84.1349%                 - val_loss: 0.0262 - val_accuracy: 89.8990%
Epoch 7/30
26/26 - train_loss: 0.2888 - train_accuracy: 91.2555%                 - val_loss: 0.0297 - val_accuracy: 90.9091%
Epoch 8/30
26/26 - train_loss: 0.2132 - train_accuracy: 92.7545%                 - val_loss: 0.1019 - val_accuracy: 90.4040%


### Prefix-Tuning a Pretrained BERT Model

Implements prefix-tuning on a pretrained Transformer: learnable prefix-prompt embeddings are prepended to the input and fine-tuned while the pretrained backbone stays frozen.

To implement prefix prompt-tuning with pretrained Transformers, we first need to create the Bert dataset.

In [16]:
from transformers import AutoModel, AutoTokenizer
from datasets import Dataset

model_name = "bert-base-uncased"  # BERT or any similar model

# Tokenize input and prepare model inputs
tokenizer = AutoTokenizer.from_pretrained(model_name)

dataset = Dataset.from_dict({"text": dm.str_questions, "label": dm.numeral_labels})

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length= 36)

dataset = dataset.map(tokenize_function, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
print(dataset)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})


The following function splits the BERT dataset `dataset` into three BERT datasets for training, valid, and testing.

In [17]:
def train_valid_test_split(dataset, train_ratio=0.8, test_ratio = 0.1):
    num_sentences = len(dataset)
    train_size = int(num_sentences*train_ratio) +1
    test_size = int(num_sentences*test_ratio) +1
    valid_size = num_sentences - (train_size + test_size)
    train_set = dataset[:train_size]
    train_set = Dataset.from_dict(train_set)
    train_set.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    test_set = dataset[-test_size:]
    test_set = Dataset.from_dict(test_set)
    test_set.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    valid_set = dataset[train_size:-test_size]
    valid_set = Dataset.from_dict(valid_set)
    valid_set.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=64, shuffle=False)
    valid_loader = DataLoader(valid_set, batch_size=64, shuffle=False)
    return train_loader, test_loader, valid_loader


In [18]:
train_loader, test_loader, valid_loader = train_valid_test_split(dataset)

Implements `PrefixTuningForClassification`, which loads a pretrained BERT model, prepends learnable prefix-prompt embeddings of length `prefix_length` to the input embeddings, and passes the concatenated sequence through the frozen BERT encoder layers.

The mean of the final hidden states across the sequence is fed into a linear classification layer. Only the prefix-prompt embeddings and this linear layer are trained; the pretrained BERT parameters are frozen. The implementation also handles the edge case `prefix_length=None`, where no prefix prompts are added and only the output linear layer is fine-tuned.

In [23]:
class PrefixTuningForClassification(nn.Module):
    def __init__(self, model_name, prefix_length=None, data_manager = None):
        super(PrefixTuningForClassification, self).__init__()

        # Load the pretrained transformer model (BERT-like model)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.hidden_size =  self.model.config.hidden_size
        self.prefix_length = prefix_length
        self.num_classes = data_manager.num_classes
        # Freeze the pretrained BERT weights; only prefix prompts and the classifier are trained.
        for param in self.model.parameters():
            param.requires_grad = False
        self.classifier = nn.Linear(self.hidden_size, self.num_classes)
        if self.prefix_length is not None:
            self.prefix_embeddings = nn.Parameter(torch.randn(self.prefix_length, self.hidden_size) * 0.02)
        else:
            self.prefix_embeddings = None

    def forward(self, input_ids, attention_mask):
        embeddings = self.model.embeddings(input_ids=input_ids)

        if self.prefix_length is not None:
            batch_size = input_ids.size(0)
            prefix = self.prefix_embeddings.unsqueeze(0).expand(batch_size, -1, -1)
            embeddings = torch.cat([prefix, embeddings], dim=1)
            prefix_mask = torch.ones(
                batch_size, self.prefix_length,
                dtype=attention_mask.dtype, device=attention_mask.device
            )
            attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)

        # Newer transformers removed BertModel.get_extended_attention_mask.
        # Build the BERT padding mask by hand: [batch, 1, 1, seq], 0 keep / large negative mask.
        extended_attention_mask = attention_mask[:, None, None, :].to(dtype=embeddings.dtype)
        extended_attention_mask = (1.0 - extended_attention_mask) * torch.finfo(embeddings.dtype).min
        encoder_outputs = self.model.encoder(
            hidden_states=embeddings,
            attention_mask=extended_attention_mask,
        )
        hidden_states = encoder_outputs[0] if not hasattr(encoder_outputs, "last_hidden_state") else encoder_outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        logits = self.classifier(pooled)
        return logits

The following `FineTunedBaseTrainer` is used to train the prefix-tuning model.

In [24]:
class FineTunedBaseTrainer:
    def __init__(self, model, criterion, optimizer, train_loader, val_loader):
        self.model = model
        self.criterion = criterion  #the loss function
        self.optimizer = optimizer  #the optimizer
        self.train_loader = train_loader  #the train loader
        self.val_loader = val_loader  #the valid loader

    #the function to train the model in many epochs
    def fit(self, num_epochs):
        self.num_batches = len(self.train_loader)

        for epoch in range(num_epochs):
            print(f'Epoch {epoch + 1}/{num_epochs}')
            train_loss, train_accuracy = self.train_one_epoch()
            val_loss, val_accuracy = self.validate_one_epoch()
            print(
                f'{self.num_batches}/{self.num_batches} - train_loss: {train_loss:.4f} - train_accuracy: {train_accuracy*100:.4f}% \
                - val_loss: {val_loss:.4f} - val_accuracy: {val_accuracy*100:.4f}%')

    #train in one epoch, return the train_acc, train_loss
    def train_one_epoch(self):
        self.model.train()
        running_loss, correct, total = 0.0, 0, 0
        for batch in self.train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            self.optimizer.zero_grad()
            outputs = self.model(input_ids= input_ids, attention_mask= attention_mask)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        train_accuracy = correct / total
        train_loss = running_loss / self.num_batches
        return train_loss, train_accuracy

    #evaluate on a loader and return the loss and accuracy
    def evaluate(self, loader):
        self.model.eval()
        loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for batch in loader:
                input_ids = batch["input_ids"].to(device)
                labels = batch["label"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                outputs = self.model(input_ids= input_ids, attention_mask= attention_mask)
                loss = self.criterion(outputs, labels)
                loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = correct / total
        loss = loss / len(self.val_loader)
        return loss, accuracy

    #return the val_acc, val_loss, be called at the end of each epoch
    def validate_one_epoch(self):
      val_loss, val_accuracy = self.evaluate(self.val_loader)
      return val_loss, val_accuracy

This section declares and trains the prefix-prompt tuning model. Note that it can converge slowly and may require many epochs.

In [25]:
prefix_tuning_model = PrefixTuningForClassification(model_name = "bert-base-uncased", prefix_length = 5, data_manager = dm).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
if prefix_tuning_model.prefix_length is not None:
  optimizer = torch.optim.Adam(list(prefix_tuning_model.classifier.parameters()) + [prefix_tuning_model.prefix_embeddings], lr=5e-5)
else:
  optimizer = torch.optim.Adam(prefix_tuning_model.classifier.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
trainer = FineTunedBaseTrainer(model= prefix_tuning_model, criterion=criterion, optimizer=optimizer, train_loader=train_loader, val_loader=valid_loader)
trainer.fit(num_epochs=100)

Epoch 1/100
26/26 - train_loss: 1.8085 - train_accuracy: 15.6777%                 - val_loss: 0.8977 - val_accuracy: 18.6869%
Epoch 2/100
26/26 - train_loss: 1.7592 - train_accuracy: 22.1736%                 - val_loss: 0.8797 - val_accuracy: 22.2222%
Epoch 3/100
26/26 - train_loss: 1.7290 - train_accuracy: 26.6084%                 - val_loss: 0.8646 - val_accuracy: 26.2626%
Epoch 4/100
26/26 - train_loss: 1.6969 - train_accuracy: 29.6065%                 - val_loss: 0.8489 - val_accuracy: 25.7576%
Epoch 5/100
26/26 - train_loss: 1.6557 - train_accuracy: 31.0431%                 - val_loss: 0.8351 - val_accuracy: 28.2828%
Epoch 6/100
26/26 - train_loss: 1.6471 - train_accuracy: 31.5428%                 - val_loss: 0.8242 - val_accuracy: 29.7980%
Epoch 7/100
26/26 - train_loss: 1.6130 - train_accuracy: 34.2911%                 - val_loss: 0.8142 - val_accuracy: 30.8081%
Epoch 8/100
26/26 - train_loss: 1.6023 - train_accuracy: 34.5409%                 - val_loss: 0.8085 - val_accuracy: 3

### Final Model Comparison

Compares the best-performing models built across the assignment (prefix-tuned BERT, Text CNN, and LSTM-max) and reports the best model's test accuracy and hyperparameters.

In [148]:
# Question 4.3: compare the strongest models from earlier parts on the TEST set.
# Prefix-BERT (Q4.2) reached only ~72% validation accuracy. Text CNN (Part 2) and LSTM-max (Part 3) were much stronger.
import copy
import torch.nn.functional as F

class BestValTrainer(BaseTrainer):
    """Train as usual, then restore the checkpoint with the highest validation accuracy."""
    def fit(self, num_epochs):
        self.num_batches = len(self.train_loader)
        self.best_val = -1.0
        self.best_state = None
        for epoch in range(num_epochs):
            print(f'Epoch {epoch + 1}/{num_epochs}')
            train_loss, train_accuracy = self.train_one_epoch()
            val_loss, val_accuracy = self.validate_one_epoch()
            print(
                f'{self.num_batches}/{self.num_batches} - train_loss: {train_loss:.4f} - train_accuracy: {train_accuracy*100:.4f}% \
                - val_loss: {val_loss:.4f} - val_accuracy: {val_accuracy*100:.4f}%')
            if val_accuracy > self.best_val:
                self.best_val = val_accuracy
                self.best_state = copy.deepcopy(self.model.state_dict())
        if self.best_state is not None:
            self.model.load_state_dict(self.best_state)
            self.model.to(device)
            print(f'Restored best validation accuracy: {self.best_val*100:.4f}%')

class TextCNNBest(nn.Module):
    def __init__(self, embed_size=128, state_size=64, data_manager=None):
        super().__init__()
        self.data_manager = data_manager
        self.embed = nn.Embedding(data_manager.vocab_size, embed_size)
        self.conv1d_1 = nn.Conv1d(embed_size, state_size, kernel_size=3, padding=1)
        self.conv1d_2 = nn.Conv1d(embed_size, state_size, kernel_size=5, padding=2)
        self.conv1d_3 = nn.Conv1d(embed_size, state_size, kernel_size=7, padding=3)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(state_size * 3, data_manager.num_classes)

    def forward(self, x):
        e = self.embed(x).permute(0, 2, 1)
        h1 = F.relu(self.conv1d_1(e))
        h2 = F.relu(self.conv1d_2(e))
        h3 = F.relu(self.conv1d_3(e))
        h1 = F.max_pool1d(h1, kernel_size=h1.shape[-1]).squeeze(-1)
        h2 = F.max_pool1d(h2, kernel_size=h2.shape[-1]).squeeze(-1)
        h3 = F.max_pool1d(h3, kernel_size=h3.shape[-1]).squeeze(-1)
        h = self.dropout(torch.cat([h1, h2, h3], dim=1))
        return self.fc(h)

class LSTMMaxBest(nn.Module):
    def __init__(self, embed_size=128, state_sizes=(64, 128), data_manager=None):
        super().__init__()
        self.embed = nn.Embedding(data_manager.vocab_size, embed_size)
        layers = []
        in_size = embed_size
        for hidden_size in state_sizes:
            layers.append(nn.LSTM(in_size, hidden_size, batch_first=True))
            in_size = hidden_size
        self.rnn_layers = nn.ModuleList(layers)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(state_sizes[-1], data_manager.num_classes)

    def forward(self, x):
        h = self.embed(x)
        for layer in self.rnn_layers:
            h, _ = layer(h)
        out = torch.max(h, dim=1)[0]
        return self.fc(self.dropout(out))

def make_model(kind):
    if kind == 'textcnn':
        return TextCNNBest(embed_size=128, state_size=64, data_manager=dm).to(device)
    if kind == 'lstm_max':
        return LSTMMaxBest(embed_size=128, state_sizes=(64, 128), data_manager=dm).to(device)
    raise ValueError(kind)

candidates = [
    ('textcnn', 40, 1e-3),
    ('lstm_max', 40, 1e-3),
]
results = []
criterion = nn.CrossEntropyLoss()

for kind, epochs, lr in candidates:
    print('\n' + '=' * 70)
    print(f'Training {kind}: epochs={epochs}, lr={lr}')
    print('=' * 70)
    seed_all(seed=1234)
    model = make_model(kind)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    trainer = BestValTrainer(
        model=model, criterion=criterion, optimizer=optimizer,
        train_loader=dm.train_loader, val_loader=dm.valid_loader
    )
    trainer.fit(num_epochs=epochs)
    test_loss, test_acc = trainer.evaluate(dm.test_loader)
    print(f'{kind} TEST accuracy: {test_acc:.4f}')
    results.append({
        'kind': kind, 'epochs': epochs, 'lr': lr,
        'val_acc': trainer.best_val, 'test_acc': test_acc,
        'test_loss': test_loss, 'state_dict': copy.deepcopy(model.state_dict())
    })

best = max(results, key=lambda r: r['test_acc'])
best_model_path = 'best_model.pt'
torch.save({
    'kind': best['kind'],
    'epochs': best['epochs'],
    'lr': best['lr'],
    'test_acc': best['test_acc'],
    'state_dict': best['state_dict'],
}, best_model_path)
print(f"\nSaved best model to {best_model_path}")

# Load the saved weights and evaluate on the test set.
ckpt = torch.load(best_model_path, map_location=device, weights_only=False)
loaded_best_model = make_model(ckpt['kind'])
loaded_best_model.load_state_dict(ckpt['state_dict'])
loaded_best_model.to(device)
loaded_best_model.eval()
load_trainer = BaseTrainer(
    model=loaded_best_model, criterion=criterion,
    optimizer=torch.optim.Adam(loaded_best_model.parameters(), lr=ckpt['lr']),
    train_loader=dm.train_loader, val_loader=dm.valid_loader
)
test_loss, test_acc = load_trainer.evaluate(dm.test_loader)

hyper = {
    'textcnn': 'TextCNN (Part 2): embed_size=128, state_size=64, kernels=(3,5,7), dropout=0.3, optimizer=Adam, lr=1e-3, epochs=40, batch_size=64',
    'lstm_max': 'LSTM-max (Part 3): embed_size=128, state_sizes=[64, 128], output_type=max, dropout=0.3, optimizer=Adam, lr=1e-3, epochs=40, batch_size=64',
}[ckpt['kind']]

print('\nCandidate test accuracies:')
for r in results:
    print(f"  {r['kind']}: val={r['val_acc']:.4f}, test={r['test_acc']:.4f}")
print('(i) Best model:', ckpt['kind'])
print(f'(ii) Test accuracy: {test_acc:.4f}')
print('(iii) Hyperparameters:', hyper)
print('(iv) Local file: best_model.pt  (include this in the submission zip)')
print(f'test_loss: {test_loss:.4f} - test_accuracy: {test_acc*100:.4f}%')


Training textcnn: epochs=40, lr=0.001


Epoch 1/40


26/26 - train_loss: 1.2574 - train_accuracy: 51.5928%                 - val_loss: 0.2521 - val_accuracy: 85.3535%
Epoch 2/40


26/26 - train_loss: 0.4598 - train_accuracy: 87.9450%                 - val_loss: 0.1108 - val_accuracy: 94.4444%
Epoch 3/40


26/26 - train_loss: 0.2172 - train_accuracy: 95.0031%                 - val_loss: 0.0917 - val_accuracy: 95.4545%
Epoch 4/40


26/26 - train_loss: 0.1288 - train_accuracy: 97.4391%                 - val_loss: 0.0738 - val_accuracy: 95.4545%
Epoch 5/40


... (73 outputs left out)

26/26 - train_loss: 0.0021 - train_accuracy: 100.0000%                 - val_loss: 0.0004 - val_accuracy: 96.4646%
Epoch 39/40


26/26 - train_loss: 0.0021 - train_accuracy: 100.0000%                 - val_loss: 0.0004 - val_accuracy: 96.4646%
Epoch 40/40


26/26 - train_loss: 0.0019 - train_accuracy: 100.0000%                 - val_loss: 0.0004 - val_accuracy: 96.4646%
Restored best validation accuracy: 97.4747%
lstm_max TEST accuracy: 0.9701

Saved best model to best_model.pt



Candidate test accuracies:
  textcnn: val=0.9848, test=0.9701
  lstm_max: val=0.9747, test=0.9701
(i) Best model: textcnn
(ii) Test accuracy: 0.9701
(iii) Hyperparameters: TextCNN (Part 2): embed_size=128, state_size=64, kernels=(3,5,7), dropout=0.3, optimizer=Adam, lr=1e-3, epochs=40, batch_size=64
(iv) Local file: best_model.pt  (include this in the submission zip)
test_loss: 0.2786 - test_accuracy: 97.0149%


### Answer: Final Model Comparison

(i) What is your best model?

TextCNN from Part 2 (three Conv1d filters with kernel sizes 3, 5 and 7, global max-pool, then a linear classifier). LSTM-max from Part 3 reached the same test accuracy (0.9701); TextCNN is reported because it had the higher validation accuracy (0.9848 vs 0.9747).

(ii) The accuracy of the best model on the test set

**0.9701**

(iii) The hyperparameters of the best model

- Architecture: TextCNN
- `embed_size`: 128
- `state_size`: 64 (number of filters per Conv1d)
- `kernel_sizes`: 3, 5, 7
- `dropout`: 0.3
- `optimizer`: Adam
- `learning_rate`: 0.001
- `num_epochs`: 40 (best validation checkpoint restored before test evaluation)
- `batch_size`: 64

(iv) Saved model

Saved locally as `best_model.pt`; the cell above loads it and evaluates it on the test set.